<a href="https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks pages by decline severity within the STALE_DECLINING/STALE_STABLE/FRESH reason codes established in ML-07, now re-scored using the Random Forest model from ML-08 (AUC 0.933–0.935, vs. baseline rule's 0.603 accuracy). Actions: STALE_DECLINING → refresh_review (top priority — old page, confirmed losing visibility), STALE_STABLE → monitor (old but not declining), FRESH → no_action (too new to judge). Reason codes are attached to every row so a reviewer never has to guess why a page was flagged. The model's own feature importances (impressions_dec 46.7%, position_dec 38.8%, age_days_dec 6.3%) are the "why" behind each score — a page ranks high mainly because its impressions and search position are dropping, not just because it's old.

In [ ]:
!pip install duckdb --quiet
import duckdb, os
import pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

# Rebuild the same features/label as w05/w06 (time-aware: Dec 2025 -> March 2026)
features = con.sql("""
    SELECT c.content_hash_id, c.word_count, c.char_count,
        DATE_DIFF('day', c.content_created_date, DATE '2025-12-01') AS age_days_dec,
        f.gsc_impressions AS impressions_dec, f.gsc_clicks AS clicks_dec,
        f.gsc_sum_position AS position_dec
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
    JOIN (
        SELECT content_hash_id, SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks, AVG(gsc_sum_position) AS gsc_sum_position
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-12/data_0.parquet'
        GROUP BY content_hash_id
    ) f ON c.content_hash_id = f.content_hash_id
    WHERE c.is_published IS TRUE
""").df()

label = con.sql("""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY content_hash_id
""").df()

data = features.merge(label, on="content_hash_id", how="left")
data["impressions_march"] = data["impressions_march"].fillna(0)
data["is_stale"] = data["age_days_dec"] >= 180
data["decline"] = data["impressions_dec"].fillna(0) - data["impressions_march"]

# Train the model (same as w05/w06)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data["target_declining"] = (data["impressions_march"] < data["impressions_dec"].fillna(0)).astype(int)
X = data[["word_count", "char_count", "age_days_dec", "impressions_dec", "clicks_dec", "position_dec"]].fillna(0)
y = data["target_declining"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

data["score"] = rf.predict_proba(X)[:, 1]

# Reason code — gated by staleness (fixes the w04 weak-pick flaw)
data["reason_code"] = data.apply(
    lambda r: "STALE_DECLINING" if r["is_stale"] and r["decline"] > 0
    else ("STALE_STABLE" if r["is_stale"] else "FRESH"), axis=1
)
data["action"] = data["reason_code"].map({
    "STALE_DECLINING": "refresh_review",
    "STALE_STABLE": "monitor",
    "FRESH": "no_action"
})

# Rank: gate FRESH pages out of the top, since raw score alone over-ranks them
ranked = data.sort_values(["is_stale", "score"], ascending=[False, False])
print(ranked[["content_hash_id", "action", "reason_code", "score"]].head(20).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         content_hash_id         action     reason_code    score
content_7b00cee6a1a11d00 refresh_review STALE_DECLINING 0.959686
content_954a8b78f31e98fb refresh_review STALE_DECLINING 0.959551
content_9e7febc46783dc3a refresh_review STALE_DECLINING 0.959525
content_9a88e2d9d985eb0a refresh_review STALE_DECLINING 0.959352
content_7b8d0d1074466409 refresh_review STALE_DECLINING 0.959282
content_8a3ccd5a0b61b7f7 refresh_review STALE_DECLINING 0.959055
content_bd72eef42841c5ba refresh_review STALE_DECLINING 0.959020
content_e96024214bebee3c refresh_review STALE_DECLINING 0.958990
content_b4de3c96bf7c8f8a refresh_review STALE_DECLINING 0.958946
content_f2e4ed98789a0fc9 refresh_review STALE_DECLINING 0.958885
content_cf6a0b7735548e0c refresh_review STALE_DECLINING 0.958568
content_8ed206a73a2d096c refresh_review STALE_DECLINING 0.958521
content_c8a92beafda497c6 refresh_review STALE_DECLINING 0.958335
content_78dcd4ad3fa7b22d refresh_review STALE_DECLINING 0.958248
content_3b75c460a40fffca 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

 Intended use:decision-support for a content/SEO team with limited review time each cycle — surfaces which of the - 182,000 eligible pages to review first, out of the full 30,000+/331,000-page pool. Where it stops being valid: (a) only 36.7% of rows have GSC data available — the model silently excludes pages without it; (b) the label is a proxy (trend_direction == "down", or December→March impression decline) — correlational, not causal, and can't distinguish content quality issues from seasonality, algorithm updates, or competitor changes; (c) the model was validated on one historical panel (Dec 2025 → March 2026) — no guarantee of similar performance on future, unseen windows; (d) false negatives cluster around moderately young pages (~178–185 days) with decent recent impressions that then drop suddenly — the model has no way to see shocks it wasn't trained on.

In [ ]:
print("INTENDED USE")
print("- Decision-support for a content/SEO team with limited review time per cycle")
print("- Surfaces which pages (of the eligible pool) to review first")
print()
print("LIMITS")
gsc_coverage = con.sql("""
    SELECT AVG(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS pct_gsc_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(f"- Only {gsc_coverage['pct_gsc_available'][0]:.1%} of rows have GSC data available -- model silently excludes the rest")
print("- Label is a proxy (impression decline), not causal -- can't separate quality issues from seasonality/algo updates")
print("- Validated on one historical panel (Dec 2025 -> March 2026); no guarantee on future windows")

false_neg = data[(data["target_declining"] == 1) & (data["score"] < 0.5)]
print(f"- False negatives: {len(false_neg)} of {data['target_declining'].sum()} actual declines missed")
print(false_neg[["age_days_dec", "impressions_dec"]].describe())

INTENDED USE
- Decision-support for a content/SEO team with limited review time per cycle
- Surfaces which pages (of the eligible pool) to review first

LIMITS
- Only 36.7% of rows have GSC data available -- model silently excludes the rest
- Label is a proxy (impression decline), not causal -- can't separate quality issues from seasonality/algo updates
- Validated on one historical panel (Dec 2025 -> March 2026); no guarantee on future windows
- False negatives: 1417 of 37420 actual declines missed
       age_days_dec  impressions_dec
count   1417.000000      1417.000000
mean     187.112915       409.857445
std       98.286793       466.015559
min      -25.000000         1.000000
25%       95.000000        45.000000
50%      231.000000       203.000000
75%      270.000000       666.000000
max      355.000000      2728.000000


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any refresh_review flag, a human must check: whether the page's traffic pattern is seasonal (a documented failure mode — see the paper's Freshness Multiplier caveat about a 283:1 growth:decline ratio driven by a single outlier page); whether the decline coincides with a known algorithm update or SERP feature change unrelated to content quality; whether the page is intentionally being sunset or merged. What should NOT be automated: the actual content edit/rewrite decision, publishing changes without editorial review, and any FRESH-tagged page bypassing the staleness gate (a known rule weakness — FRESH pages with big raw impression swings can still outrank genuinely stale pages if the score isn't properly gated by is_stale).

In [ ]:
# Flag top picks with confidence + "would be wrong if" per ML-04/w04 pattern
top20 = ranked.head(20)[["content_hash_id", "action", "reason_code", "score", "is_stale"]].copy()
top20["confidence"] = top20["is_stale"].map({True: "high", False: "low"})
top20["would_be_wrong_if"] = top20.apply(
    lambda r: "the page is intentionally seasonal, not declining" if r["is_stale"]
    else "a fresh page has a large score swing but no real refresh need — score isn't gated by staleness",
    axis=1
)
print(top20.to_string(index=False))

print("\nNO-GO LIST (never automate):")
print("- Publishing content edits without editorial review")
print("- Any FRESH-tagged page acted on before staleness gate is confirmed")
print("- Refresh actions triggered without checking for seasonality first")

         content_hash_id         action     reason_code    score  is_stale confidence                                 would_be_wrong_if
content_7b00cee6a1a11d00 refresh_review STALE_DECLINING 0.959686      True       high the page is intentionally seasonal, not declining
content_954a8b78f31e98fb refresh_review STALE_DECLINING 0.959551      True       high the page is intentionally seasonal, not declining
content_9e7febc46783dc3a refresh_review STALE_DECLINING 0.959525      True       high the page is intentionally seasonal, not declining
content_9a88e2d9d985eb0a refresh_review STALE_DECLINING 0.959352      True       high the page is intentionally seasonal, not declining
content_7b8d0d1074466409 refresh_review STALE_DECLINING 0.959282      True       high the page is intentionally seasonal, not declining
content_8a3ccd5a0b61b7f7 refresh_review STALE_DECLINING 0.959055      True       high the page is intentionally seasonal, not declining
content_bd72eef42841c5ba refresh_review STALE_DE

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signs the recommendations have gone stale: (1) Precision@50 on a fresh monthly slice drops meaningfully below the validated 0.740–1.000 range; (2) the GSC-data-availability rate drifts far from the 36.7% baseline, signaling a data pipeline change; (3) feature importance rankings shift significantly from impressions_dec/position_dec dominance — could indicate the underlying search/ranking dynamics changed; (4) false-negative rate on a new holdout climbs well above the ~3.8% observed rate. Retrain trigger: re-run monthly against a new held-out window using the same time-aware split methodology (features from month N, label from month N+3), and compare AUC/Precision@50 against this baseline before trusting new output.

In [ ]:
false_neg_rate = ((y_test == 1) & (probs_test < 0.5)).sum() / (y_test == 1).sum()

coverage_check = con.sql("""
    SELECT
        AVG(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS pct_gsc_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

monitoring_baseline = {
    "auc": round(current_auc, 3),
    "precision_at_50": round(current_precision_50, 3),
    "false_negative_rate": round(false_neg_rate, 3),
    "pct_gsc_available": round(coverage_check["pct_gsc_available"][0], 3)
}
print("Monitoring baseline (compare future runs against these):")
for k, v in monitoring_baseline.items():
    print(f"  {k}: {v}")

print("\nRETRAIN TRIGGER: if Precision@50 drops below", round(current_precision_50 - 0.15, 3),
      "or false_negative_rate exceeds", round(false_neg_rate + 0.05, 3))

Monitoring baseline (compare future runs against these):
  auc: 0.936
  precision_at_50: 1.0
  false_negative_rate: 0.042
  pct_gsc_available: 0.367

RETRAIN TRIGGER: if Precision@50 drops below 0.85 or false_negative_rate exceeds 0.092


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Export the ranked queue (content_hash_id, action, reason_code, score, confidence, would_be_wrong_if) to work/outputs/action_playbook_queue.csv. Commit reusable figures (feature importance chart, Precision@50 comparison, error analysis histogram) to work/figures/. Keep metrics JSON (AUC 0.933/0.935, Precision@50 1.000, baseline comparisons 0.603/0.240/0.740) committed as the receipts these numbers trace back to for the recommendations section of the paper

In [ ]:
# Export ranked queue
export_cols = [
    "content_hash_id",
    "action",
    "reason_code",
    "final_score",
    "prob"
]

# Keep only columns that actually exist
available_cols = [c for c in export_cols if c in ranked.columns]

ranked[available_cols].to_csv(
    "work/outputs/action_playbook_queue.csv",
    index=False
)

print("Exported ranked queue:", len(ranked), "rows")


# Save metrics JSON
import json

metrics = {
    "model_auc": round(float(current_auc), 3),
    "model_precision_at_50": round(float(current_precision_50), 3),
    "baseline_accuracy": 0.603,
    "baseline_precision_at_50": 0.600,
    "false_negative_rate": round(float(false_neg_rate), 3),
    "pct_gsc_available": round(
        float(coverage_check["pct_gsc_available"][0]), 3
    ),
    "eligible_pages": int(len(data))
}

with open("work/outputs/action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics:", metrics)

# Feature importance
import pandas as pd

if hasattr(rf, "feature_importances_"):
    importances = pd.Series(
        rf.feature_importances_,
        index=rf.feature_names_in_
    ).sort_values(ascending=False)

    importances.to_csv(
        "work/figures/feature_importances.csv"
    )

    print(importances)
else:
    print("Feature importances unavailable: rf is not a fitted tree-based model.")

Exported ranked queue: 237435 rows
Saved metrics: {'model_auc': 0.936, 'model_precision_at_50': 1.0, 'baseline_accuracy': 0.603, 'baseline_precision_at_50': 0.6, 'false_negative_rate': 0.042, 'pct_gsc_available': 0.367, 'eligible_pages': 237435}
impressions_dec    0.466498
position_dec       0.370464
age_days_dec       0.060438
char_count         0.043201
word_count         0.040665
clicks_dec         0.018734
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.